<a href="https://colab.research.google.com/github/edwojciecho-code/healthcare-access-dashboard-poland/blob/main/czyszczenie_danych.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

In [ ]:
# Pokazujemy jakie pliki wczytal z dysku
csv_files_path = '/content/drive/My Drive/Projekt/'
all_csv_files = glob.glob(os.path.join(csv_files_path, '*.csv'))

for f in all_csv_files:
    print(f)

/content/drive/My Drive/Projekt/szpitale_lozka_ogolem.csv
/content/drive/My Drive/Projekt/pielegniarki_polozne.csv
/content/drive/My Drive/Projekt/lekarze.csv
/content/drive/My Drive/Projekt/apteki.csv
/content/drive/My Drive/Projekt/punkty_apteczne.csv
/content/drive/My Drive/Projekt/osoby_swiadczenia_spoleczne.csv
/content/drive/My Drive/Projekt/ambulatoryjna_opieka_placowki.csv
/content/drive/My Drive/Projekt/ludnosc.csv
/content/drive/My Drive/Projekt/lozka.csv


In [ ]:
list_of_processed_dfs = []

# Kolumny po których będziemy łączyć tabele
id_vars = ['Kod', 'Nazwa']

for filename in all_csv_files:
    try:
        # Pliki z GUS -> separator średnikowy
        df = pd.read_csv(filename, sep=';')

        # Nazwa wskaznika po nazwie pliku na podstawie biblioteki os
        metric_name = os.path.splitext(os.path.basename(filename))[0]

        # Wyszukanie kolumn, które zawierają rok
        year_columns = [
            col for col in df.columns
            if col not in id_vars and re.search(r'\d{4}', str(col))
        ]

        # Zamiana danych z układu kod, nazwa, rok1, rok2 na kod, nazwa, rok, wskaznik
        df_melted = df.melt(
            id_vars=id_vars,
            value_vars=year_columns,
            var_name='kolumna_roku',
            value_name=metric_name
        )

        # Wyciągnięcie roku z nazwy kolumny
        df_melted['Year'] = (
            df_melted['kolumna_roku']
            .astype(str)
            .str.extract(r'(\d{4})')[0]
        )

        # Usunięcie wierszy, w których nie udało się odczytać roku
        df_melted = df_melted.dropna(subset=['Year'])

        # Kolumna z oryginalną nazwą roku nie jest już potrzebna
        df_melted = df_melted.drop(columns='kolumna_roku')

        # Zamieniamy przecinki na kropki
        df_melted[metric_name] = (
            df_melted[metric_name]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.replace(' ', '', regex=False)
        )

        df_melted[metric_name] = pd.to_numeric(
            df_melted[metric_name],
            errors='coerce'
        )

        # Jeżeli pojawiły się duplikaty dla tego samego regionu i roku zostawiamy pierwszą wartość
        df_melted = (
            df_melted
            .groupby(id_vars + ['Year'], as_index=False)
            .first()
        )

        # Dodanie przetworzonej tabeli do listy
        list_of_processed_dfs.append(df_melted)

    except Exception as e:
        print(f"Błąd")


# Łączenie wszystkich tabel w jedną
if list_of_processed_dfs:
    combined_df = list_of_processed_dfs[0]

    # Łączymy tabele po kodzie regionu, nazwie regionu i roku
    for df_to_merge in list_of_processed_dfs[1:]:
        combined_df = pd.merge(
            combined_df,
            df_to_merge,
            on=['Kod', 'Nazwa', 'Year'],
            how='inner'
        )
    display(combined_df.head())

,Kod,Nazwa,Year,szpitale_lozka_ogolem,pielegniarki_polozne,lekarze,apteki,punkty_apteczne,osoby_swiadczenia_spoleczne,ambulatoryjna_opieka_placowki,ludnosc,lozka
0,0,POLSKA,2006,742.0,193030.0,145181.0,10334.0,1042.0,NaN,13473.0,38125479.0,176673.0
1,0,POLSKA,2007,748.0,199391.0,150115.0,10625.0,1114.0,NaN,14206.0,38115641.0,175023.0
2,0,POLSKA,2008,732.0,203261.0,154428.0,10623.0,1103.0,NaN,14853.0,38135876.0,183565.0
3,0,POLSKA,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
4,0,POLSKA,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0


In [ ]:
# Zostawiamy tylko wiersze z latami 2009-2024
combined_df['Year'] = pd.to_numeric(combined_df['Year'], errors='coerce')
combined_df = combined_df[(combined_df['Year'] >= 2009) & (combined_df['Year'] <= 2024)]

display(combined_df)
print(f"wiersze {len(combined_df)}")
print(f"kolumny {len(combined_df.columns)}")

,Kod,Nazwa,Year,szpitale_lozka_ogolem,pielegniarki_polozne,lekarze,apteki,punkty_apteczne,osoby_swiadczenia_spoleczne,ambulatoryjna_opieka_placowki,ludnosc,lozka
3,0,POLSKA,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
4,0,POLSKA,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0
5,0,POLSKA,2011,814.0,216462.0,172304.0,11713.0,1185.0,2691933.0,19151.0,38538447.0,180606.0
6,0,POLSKA,2012,913.0,237862.0,183906.0,11999.0,1220.0,2594317.0,19412.0,38533299.0,188820.0
7,0,POLSKA,2013,966.0,227499.0,187327.0,12221.0,1284.0,2709649.0,19529.0,38495659.0,187763.0
...,...,...,...,...,...,...,...,...,...,...,...,...
334,3200000,ZACHODNIOPOMORSKIE,2020,44.0,9761.0,8880.0,549.0,40.0,73007.0,884.0,1661073.0,6901.0
335,3200000,ZACHODNIOPOMORSKIE,2021,44.0,10337.0,5831.0,541.0,41.0,65911.0,941.0,1650021.0,7052.0
336,3200000,ZACHODNIOPOMORSKIE,2022,44.0,8051.0,5544.0,533.0,45.0,59315.0,997.0,1640622.0,6809.0
337,3200000,ZACHODNIOPOMORSKIE,2023,46.0,8483.0,6317.0,531.0,41.0,57345.0,1024.0,1631784.0,6631.0


wiersze 272
kolumny 12


In [ ]:
#Usuwamy CapsLock'a z kolumny Nazwa
combined_df['Nazwa'] = combined_df['Nazwa'].str.title()
display(combined_df.head())

,Kod,Nazwa,Year,szpitale_lozka_ogolem,pielegniarki_polozne,lekarze,apteki,punkty_apteczne,osoby_swiadczenia_spoleczne,ambulatoryjna_opieka_placowki,ludnosc,lozka
3,0,Polska,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
4,0,Polska,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0
5,0,Polska,2011,814.0,216462.0,172304.0,11713.0,1185.0,2691933.0,19151.0,38538447.0,180606.0
6,0,Polska,2012,913.0,237862.0,183906.0,11999.0,1220.0,2594317.0,19412.0,38533299.0,188820.0
7,0,Polska,2013,966.0,227499.0,187327.0,12221.0,1284.0,2709649.0,19529.0,38495659.0,187763.0


In [ ]:
# Zmieniamy nazwy kolumn
combined_df = combined_df.rename(columns={'Year': 'Rok'})
combined_df = combined_df.rename(columns={'szpitale_lozka_ogolem': 'Szpitale'})
combined_df = combined_df.rename(columns={'pielegniarki_polozne': 'Pielegniarki_Polozne'})
combined_df = combined_df.rename(columns={'lekarze': 'Lekarze'})
combined_df = combined_df.rename(columns={'apteki': 'Apteki'})
combined_df = combined_df.rename(columns={'punkty_apteczne': 'Punkty_apteczne'})
combined_df = combined_df.rename(columns={'osoby_swiadczenia_spoleczne': 'Osoby_swiadczenia_spoleczne'})
combined_df = combined_df.rename(columns={'ambulatoryjna_opieka_placowki': 'Placowki_ambulatoryjne'})
combined_df = combined_df.rename(columns={'ludnosc': 'Ludnosc'})
combined_df = combined_df.rename(columns={'lozka': 'Lozka'})


display(combined_df.head())

,Kod,Nazwa,Rok,Szpitale,Pielegniarki_Polozne,Lekarze,Apteki,Punkty_apteczne,Osoby_swiadczenia_spoleczne,Placowki_ambulatoryjne,Ludnosc,Lozka
3,0,Polska,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
4,0,Polska,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0
5,0,Polska,2011,814.0,216462.0,172304.0,11713.0,1185.0,2691933.0,19151.0,38538447.0,180606.0
6,0,Polska,2012,913.0,237862.0,183906.0,11999.0,1220.0,2594317.0,19412.0,38533299.0,188820.0
7,0,Polska,2013,966.0,227499.0,187327.0,12221.0,1284.0,2709649.0,19529.0,38495659.0,187763.0


In [ ]:
# Resetujemy indeksy
combined_df = combined_df.reset_index(drop=True)

In [ ]:
combined_df = combined_df.drop(columns=['Kod'])

,Nazwa,Rok,Szpitale,Pielegniarki_Polozne,Lekarze,Apteki,Punkty_apteczne,Osoby_swiadczenia_spoleczne,Placowki_ambulatoryjne,Ludnosc,Lozka
0,Polska,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
1,Polska,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0
2,Polska,2011,814.0,216462.0,172304.0,11713.0,1185.0,2691933.0,19151.0,38538447.0,180606.0
3,Polska,2012,913.0,237862.0,183906.0,11999.0,1220.0,2594317.0,19412.0,38533299.0,188820.0
4,Polska,2013,966.0,227499.0,187327.0,12221.0,1284.0,2709649.0,19529.0,38495659.0,187763.0


In [ ]:
display(combined_df)

,Nazwa,Rok,Szpitale,Pielegniarki_Polozne,Lekarze,Apteki,Punkty_apteczne,Osoby_swiadczenia_spoleczne,Placowki_ambulatoryjne,Ludnosc,Lozka
0,Polska,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
1,Polska,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0
2,Polska,2011,814.0,216462.0,172304.0,11713.0,1185.0,2691933.0,19151.0,38538447.0,180606.0
3,Polska,2012,913.0,237862.0,183906.0,11999.0,1220.0,2594317.0,19412.0,38533299.0,188820.0
4,Polska,2013,966.0,227499.0,187327.0,12221.0,1284.0,2709649.0,19529.0,38495659.0,187763.0
...,...,...,...,...,...,...,...,...,...,...,...
267,Zachodniopomorskie,2020,44.0,9761.0,8880.0,549.0,40.0,73007.0,884.0,1661073.0,6901.0
268,Zachodniopomorskie,2021,44.0,10337.0,5831.0,541.0,41.0,65911.0,941.0,1650021.0,7052.0
269,Zachodniopomorskie,2022,44.0,8051.0,5544.0,533.0,45.0,59315.0,997.0,1640622.0,6809.0
270,Zachodniopomorskie,2023,46.0,8483.0,6317.0,531.0,41.0,57345.0,1024.0,1631784.0,6631.0


In [ ]:
# Zapisujemy do pliku CSV na dysku
output_path = '/content/drive/My Drive/Projekt/dataset.csv'
combined_df.to_csv(output_path, index=False)

In [ ]:
import pandas as pd
df = pd.read_csv('dataset.csv')
display(df.head())

,Nazwa,Rok,Szpitale,Pielegniarki_Polozne,Lekarze,Apteki,Punkty_apteczne,Osoby_swiadczenia_spoleczne,Placowki_ambulatoryjne,Ludnosc,Lozka
0,Polska,2009,754.0,210999.0,159712.0,10817.0,1154.0,2786061.0,16252.0,38167329.0,183040.0
1,Polska,2010,795.0,210402.0,162522.0,11297.0,1161.0,2776999.0,16608.0,38529866.0,181077.0
2,Polska,2011,814.0,216462.0,172304.0,11713.0,1185.0,2691933.0,19151.0,38538447.0,180606.0
3,Polska,2012,913.0,237862.0,183906.0,11999.0,1220.0,2594317.0,19412.0,38533299.0,188820.0
4,Polska,2013,966.0,227499.0,187327.0,12221.0,1284.0,2709649.0,19529.0,38495659.0,187763.0


In [ ]:
print(df.columns)

Index(['Nazwa', 'Rok', 'Szpitale', 'Pielegniarki_Polozne', 'Lekarze', 'Apteki',
       'Punkty_apteczne', 'Osoby_swiadczenia_spoleczne',
       'Placowki_ambulatoryjne', 'Ludnosc', 'Lozka'],
      dtype='object')


In [ ]:
df_zdrowie = df.copy()

df_zdrowie = df_zdrowie[df_zdrowie['Nazwa'] != 'Polska']

# Obliczamy liczbę lekarzy na 10 tys. mieszkańców
df_zdrowie['lekarze_na_10tys'] = (
    df_zdrowie['Lekarze'] / df_zdrowie['Ludnosc'] * 10000
)

# Obliczamy liczbę pielęgniarek i położnych na 10 tys. mieszkańców
df_zdrowie['pielegniarki_polozne_na_10tys'] = (
    df_zdrowie['Pielegniarki_Polozne'] / df_zdrowie['Ludnosc'] * 10000
)

# Obliczamy liczbę aptek na 10 tys. mieszkańców
df_zdrowie['apteki_na_10tys'] = (
    df_zdrowie['Apteki'] / df_zdrowie['Ludnosc'] * 10000
)

# Obliczamy liczbę punktów aptecznych na 10 tys. mieszkańców
df_zdrowie['punkty_apteczne_na_10tys'] = (
    df_zdrowie['Punkty_apteczne'] / df_zdrowie['Ludnosc'] * 10000
)

# Łączny wskaźnik aptek i punktów aptecznych, aby zmierzyć ogólną dostępność usług aptecznych
df_zdrowie['apteki_i_punkty_na_10tys'] = (
    (df_zdrowie['Apteki'] + df_zdrowie['Punkty_apteczne'])
    / df_zdrowie['Ludnosc'] * 10000
)

# Obliczamy liczbę szpitali na 10 tys. mieszkańców
df_zdrowie['szpitale_na_10tys'] = (
    df_zdrowie['Szpitale'] / df_zdrowie['Ludnosc'] * 10000
)

# Obliczamy liczbę łóżek szpitalnych na 10 tys. mieszkańców
df_zdrowie['lozka_na_10tys'] = (
    df_zdrowie['Lozka'] / df_zdrowie['Ludnosc'] * 10000
)

# Obliczamy liczbę placówek ambulatoryjnych na 10 tys. mieszkańców
df_zdrowie['placowki_ambulatoryjne_na_10tys'] = (
    df_zdrowie['Placowki_ambulatoryjne'] / df_zdrowie['Ludnosc'] * 10000
)

# Dodatkowy wskaźnik: liczba mieszkańców przypadających na jedną aptekę (niższa wartość - lepiej)
df_zdrowie['mieszkancow_na_apteke'] = (
    df_zdrowie['Ludnosc'] / df_zdrowie['Apteki']
)

# Zaokrąglamy nowe wskaźniki
kolumny_wskaznikow = [
    'lekarze_na_10tys',
    'pielegniarki_polozne_na_10tys',
    'apteki_na_10tys',
    'punkty_apteczne_na_10tys',
    'apteki_i_punkty_na_10tys',
    'szpitale_na_10tys',
    'lozka_na_10tys',
    'placowki_ambulatoryjne_na_10tys',
    'mieszkancow_na_apteke'
]

df_zdrowie[kolumny_wskaznikow] = df_zdrowie[kolumny_wskaznikow].round(2)

display(df_zdrowie.head())

,Nazwa,Rok,Szpitale,Pielegniarki_Polozne,Lekarze,Apteki,Punkty_apteczne,Osoby_swiadczenia_spoleczne,Placowki_ambulatoryjne,Ludnosc,Lozka,lekarze_na_10tys,pielegniarki_polozne_na_10tys,apteki_na_10tys,punkty_apteczne_na_10tys,apteki_i_punkty_na_10tys,szpitale_na_10tys,lozka_na_10tys,placowki_ambulatoryjne_na_10tys,mieszkancow_na_apteke
16,Dolnośląskie,2009,60.0,15879.0,11846.0,922.0,59.0,192644.0,1181.0,2876627.0,13907.0,41.18,55.20,3.21,0.21,3.41,0.21,48.34,4.11,3119.99
17,Dolnośląskie,2010,67.0,15908.0,12115.0,943.0,66.0,189126.0,1178.0,2917242.0,14126.0,41.53,54.53,3.23,0.23,3.46,0.23,48.42,4.04,3093.58
18,Dolnośląskie,2011,72.0,16333.0,12889.0,969.0,69.0,184161.0,1343.0,2916577.0,14111.0,44.19,56.00,3.32,0.24,3.56,0.25,48.38,4.60,3009.88
19,Dolnośląskie,2012,80.0,18723.0,13845.0,970.0,72.0,170408.0,1369.0,2914362.0,14816.0,47.51,64.24,3.33,0.25,3.58,0.27,50.84,4.70,3004.50
20,Dolnośląskie,2013,80.0,17693.0,14132.0,993.0,75.0,176958.0,1346.0,2909997.0,15073.0,48.56,60.80,3.41,0.26,3.67,0.27,51.80,4.63,2930.51


In [ ]:
wskazniki_do_indeksu = [
    'lekarze_na_10tys',
    'pielegniarki_polozne_na_10tys',
    'apteki_i_punkty_na_10tys',
    'szpitale_na_10tys',
    'lozka_na_10tys',
    'placowki_ambulatoryjne_na_10tys'
]

# Normalizujemy wskaźniki do skali od 0 do 1
for kolumna in wskazniki_do_indeksu:
    min_wartosc = df_zdrowie[kolumna].min()
    max_wartosc = df_zdrowie[kolumna].max()

    if max_wartosc != min_wartosc:
        df_zdrowie[kolumna + '_norm'] = (
            (df_zdrowie[kolumna] - min_wartosc)
            / (max_wartosc - min_wartosc)
        )
    else:
        df_zdrowie[kolumna + '_norm'] = 0

# Lista kolumn znormalizowanych
kolumny_norm = [kolumna + '_norm' for kolumna in wskazniki_do_indeksu]

# Liczymy średnią ze znormalizowanych wskaźników to będzie nasz ogólny indeks dostępności opieki zdrowotnej
df_zdrowie['indeks_dostepnosci'] = df_zdrowie[kolumny_norm].mean(axis=1)

df_zdrowie['indeks_dostepnosci'] = df_zdrowie['indeks_dostepnosci'].round(3)

# Wyświetlamy wybrane kolumny, żeby sprawdzić wynik
display(
    df_zdrowie[
        [
            'Nazwa',
            'Rok',
            'lekarze_na_10tys',
            'pielegniarki_polozne_na_10tys',
            'apteki_i_punkty_na_10tys',
            'lozka_na_10tys',
            'placowki_ambulatoryjne_na_10tys',
            'indeks_dostepnosci'
        ]
    ].head()
)

,Nazwa,Rok,lekarze_na_10tys,pielegniarki_polozne_na_10tys,apteki_i_punkty_na_10tys,lozka_na_10tys,placowki_ambulatoryjne_na_10tys,indeks_dostepnosci
16,Dolnośląskie,2009,41.18,55.20,3.41,48.34,4.11,0.353
17,Dolnośląskie,2010,41.53,54.53,3.46,48.42,4.04,0.371
18,Dolnośląskie,2011,44.19,56.00,3.56,48.38,4.60,0.432
19,Dolnośląskie,2012,47.51,64.24,3.58,50.84,4.70,0.508
20,Dolnośląskie,2013,48.56,60.80,3.67,51.80,4.63,0.513


In [ ]:
# Tworzymy ranking województw osobno dla każdego roku
df_zdrowie['miejsce_w_rankingu'] = (
    df_zdrowie
    .groupby('Rok')['indeks_dostepnosci']
    .rank(ascending=False, method='min')
)

df_zdrowie['miejsce_w_rankingu'] = df_zdrowie['miejsce_w_rankingu'].astype(int)

display(
    df_zdrowie[
        [
            'Nazwa',
            'Rok',
            'indeks_dostepnosci',
            'miejsce_w_rankingu'
        ]
    ].sort_values(['Rok', 'miejsce_w_rankingu'])
)

,Nazwa,Rok,indeks_dostepnosci,miejsce_w_rankingu
80,Łódzkie,2009,0.497,1
48,Lubelskie,2009,0.454,2
160,Podlaskie,2009,0.451,3
192,Śląskie,2009,0.442,4
16,Dolnośląskie,2009,0.353,5
...,...,...,...,...
271,Zachodniopomorskie,2024,0.418,12
47,Kujawsko-Pomorskie,2024,0.374,13
255,Wielkopolskie,2024,0.367,14
79,Lubuskie,2024,0.342,15


In [ ]:
# Średnie miejsce każdego województwa ze wszystkich lat
ranking_sredni = (
    df_zdrowie
    .groupby('Nazwa', as_index=False)['miejsce_w_rankingu']
    .mean()
    .sort_values('miejsce_w_rankingu')
    .reset_index(drop=True)
)

ranking_sredni['miejsce_w_rankingu'] = ranking_sredni['miejsce_w_rankingu'].round(2)

display(ranking_sredni)

,Nazwa,miejsce_w_rankingu
0,Śląskie,2.00
1,Łódzkie,2.12
2,Lubelskie,2.75
3,Podlaskie,3.06
4,Dolnośląskie,5.88
5,Mazowieckie,6.56
6,Małopolskie,7.88
7,Świętokrzyskie,8.00
8,Opolskie,8.75
9,Warmińsko-Mazurskie,9.88


In [ ]:
# Porządkujemy główną tabelę
df_zdrowie = (
    df_zdrowie
    .sort_values(['Rok', 'miejsce_w_rankingu'])
    .reset_index(drop=True)
)

# Porządkujemy tabelę ze średnim rankingiem
ranking_sredni = (
    ranking_sredni
    .sort_values('miejsce_w_rankingu')
    .reset_index(drop=True)
)

df_zdrowie.to_csv(
    "zdrowie.csv",
    index=False,
    encoding="utf-8-sig"
)

ranking_sredni.to_csv(
    "srednie.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from google.colab import files

files.download("zdrowie.csv")

files.download("srednie.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>